# Lesson 2.3 — Flow Maps and Distributed Tracing

## Learning Objectives
- Read and interpret an AppDynamics Flow Map
- Understand how AppDynamics performs distributed call tracking across tiers
- Use Transaction Snapshots to trace a slow call graph
- Navigate the Service Endpoints and Remote Services views

**Reference**: [Overview of Application Monitoring](https://help.splunk.com/en/appdynamics-saas/application-performance-monitoring/26.6.0/overview-of-application-monitoring) | [Remote Services](https://help.splunk.com/en/appdynamics-saas/application-performance-monitoring/26.6.0/remote-services)

## 1. What is the Flow Map?

The **Flow Map** is AppDynamics' real-time topology visualization. It answers the question: *"How do my application's components talk to each other?"*

### What you see in a Flow Map:

```
Browser (EUM)
     │  
     ▼
[WebFrontend] ──calls──► [OrderService] ──calls──► [InventoryAPI]
     │                        │
     │                        ▼
     │                  [MySQL Backend] ◄── detected via exit calls
     │
     ▼
[CDN / External] ◄── detected HTTP call
```

| Element | Description |
|---------|-------------|
| **Tier box** | A color-coded box representing a tier (green/yellow/red) |
| **Arrow** | A call relationship, labeled with calls/min |
| **Cylinder** | A detected database backend |
| **Globe** | A remote HTTP backend (external API) |
| **Cloud** | A cloud service (S3, SQS, etc.) |
| **Queue** | A message queue (Kafka, RabbitMQ) |

In [ ]:
# Simulate a Flow Map topology programmatically

import json

# This represents what AppDynamics internally models for a Flow Map
flow_map = {
    "application": "StoreFront",
    "nodes": [
        {"name": "WebFrontend",   "type": "tier",    "health": "NORMAL",   "calls_pm": 210},
        {"name": "OrderService",  "type": "tier",    "health": "WARNING",  "calls_pm": 87},
        {"name": "InventoryAPI",  "type": "tier",    "health": "NORMAL",   "calls_pm": 87},
        {"name": "AuthService",   "type": "tier",    "health": "NORMAL",   "calls_pm": 33},
        {"name": "orders-mysql",  "type": "database","health": "NORMAL",   "calls_pm": 320},
        {"name": "redis-cache",   "type": "cache",   "health": "NORMAL",   "calls_pm": 540},
        {"name": "stripe.com",    "type": "remote",  "health": "NORMAL",   "calls_pm": 42},
    ],
    "edges": [
        {"from": "WebFrontend",  "to": "OrderService",  "calls_pm": 87,  "avg_rt_ms": 320},
        {"from": "WebFrontend",  "to": "AuthService",   "calls_pm": 33,  "avg_rt_ms": 85},
        {"from": "WebFrontend",  "to": "redis-cache",   "calls_pm": 200, "avg_rt_ms": 2},
        {"from": "OrderService", "to": "InventoryAPI",  "calls_pm": 87,  "avg_rt_ms": 95},
        {"from": "OrderService", "to": "orders-mysql",  "calls_pm": 320, "avg_rt_ms": 18},
        {"from": "OrderService", "to": "stripe.com",    "calls_pm": 42,  "avg_rt_ms": 210},
    ]
}

print(f"Flow Map: {flow_map['application']}")
print()
print("NODES:")
for node in flow_map["nodes"]:
    health_icon = {"NORMAL": "✅", "WARNING": "⚠️", "CRITICAL": "🔴"}.get(node["health"], "❓")
    print(f"  {health_icon} [{node['type']:8}] {node['name']:<20} {node['calls_pm']:>5} calls/min")

print()
print("CALL EDGES:")
for edge in flow_map["edges"]:
    print(f"  {edge['from']:<15} ─({edge['calls_pm']:>4} calls/min, {edge['avg_rt_ms']:>5}ms)→ {edge['to']}")

## 2. How AppDynamics Tracks Calls Across Tiers

AppDynamics uses **correlation headers** to propagate tracing context across services — similar to how OpenTelemetry passes trace IDs in HTTP headers.

When `WebFrontend` makes an HTTP call to `OrderService`:
1. The **WebFrontend agent** injects a correlation header: `singularityheader: <correlation-id>`
2. The **OrderService agent** reads this header and continues the same transaction
3. Both agents report their segment to the Controller with the same correlation ID
4. The Controller stitches segments together into a **distributed call graph**

```
WebFrontend (300ms total)
├── Authentication check        [15ms]
├── HTTP call → OrderService   [245ms]  ← cross-tier
│   ├── OrderService.createOrder  [80ms]
│   ├── SQL: INSERT INTO orders   [18ms]
│   └── HTTP call → stripe.com   [140ms]
└── Cache write → Redis          [2ms]
```

This gives you an exact breakdown of where each millisecond was spent.

In [ ]:
# Simulate reading a Transaction Snapshot call graph

def print_call_graph(node, indent=0):
    """Recursively print a call graph tree."""
    prefix = "  " * indent + ("└─ " if indent > 0 else "")
    duration = node.get("duration_ms", 0)
    bar = "█" * max(1, int(duration / 10))
    print(f"{prefix}{node['name']:<50} {duration:>6}ms  {bar}")
    for child in node.get("children", []):
        print_call_graph(child, indent + 1)

# A transaction snapshot for /checkout that took 847ms
snapshot = {
    "name": "HTTP GET /checkout  [WebFrontend]",
    "duration_ms": 847,
    "children": [
        {"name": "SessionValidator.validate()",       "duration_ms": 12, "children": []},
        {"name": "CartService.getCart()",              "duration_ms": 35, "children": [
            {"name": "Redis GET cart:{userId}",         "duration_ms": 3, "children": []},
        ]},
        {"name": "HTTP POST → OrderService/process",  "duration_ms": 780, "children": [
            {"name": "OrderService.processOrder()  [OrderService]", "duration_ms": 778, "children": [
                {"name": "InventoryService.reserve()  [HTTP]",       "duration_ms": 95, "children": []},
                {"name": "SQL: INSERT INTO orders...",               "duration_ms": 22, "children": []},
                {"name": "SQL: UPDATE inventory...",                 "duration_ms": 19, "children": []},
                {"name": "HTTP POST → stripe.com/charge",           "duration_ms": 625, "children": [  # ← bottleneck!
                    {"name": "[External] Stripe API call",           "duration_ms": 624, "children": []},
                ]},
            ]},
        ]},
        {"name": "Cache.invalidate()",                "duration_ms": 2, "children": []},
    ]
}

print(f"Transaction Snapshot: /checkout  (Total: {snapshot['duration_ms']}ms)")
print("-" * 70)
print_call_graph(snapshot)
print()
print("Insight: 625ms out of 847ms total was spent waiting on the Stripe API call.")
print("Root cause: External payment gateway latency.")

## 3. OpenTelemetry Integration

AppDynamics supports receiving **OpenTelemetry (OTel)** trace data through the **OTel Ingestion Gateway**.

This means:
- Applications already instrumented with OTel SDKs can send data to AppDynamics
- OTel spans appear as Transaction Snapshots and call graphs
- AppDynamics adds AI-powered baselining on top of OTel data

**Reference**: [Splunk AppDynamics for OpenTelemetry](https://help.splunk.com/en/appdynamics-saas/application-performance-monitoring/26.6.0/splunk-appdynamics-for-opentelemetry)

This is particularly useful for polyglot applications (multiple languages) or when you want to avoid vendor-specific agent instrumentation.

In [ ]:
# EXERCISE 2.3: Find the slowest call in a snapshot
#
# Given a transaction snapshot (nested call graph),
# TASK: Write a function to find the single slowest leaf node.

def find_slowest_leaf(node: dict, current_path: str = "") -> tuple:
    """
    Traverse a snapshot call graph and return (path, duration_ms) of the slowest leaf.
    A leaf is a node with no children.
    """
    path = f"{current_path} → {node['name']}" if current_path else node['name']
    
    if not node.get('children'):
        # This is a leaf node
        return (path, node.get('duration_ms', 0))
    
    # TODO: Recursively find the slowest leaf among children
    slowest = None
    # ...
    return slowest

# Test with the snapshot from above
result = find_slowest_leaf(snapshot)
if result:
    path, duration = result
    print(f"Slowest leaf call: {duration}ms")
    print(f"Path: {path}")
else:
    print("Implement find_slowest_leaf() to see the result.")

## Summary

- The **Flow Map** visualizes your entire application topology with real-time health overlays
- AppDynamics uses **correlation headers** to track a request as it flows through multiple tiers
- **Transaction Snapshots** provide a full call graph for any slow or errored transaction
- **Backends** (databases, external APIs, queues) are auto-detected and shown in the Flow Map
- AppDynamics also accepts **OpenTelemetry** trace data via the OTel Ingestion Gateway

---

**Next**: [Lesson 2.4 — Infrastructure and Database Visibility](lesson-2.4-infrastructure-database-visibility.ipynb)